# Step 2 - Advanced Models (Transformer) From Step1 Outputs

This notebook extends the baseline workflow with advanced modeling while preserving Step 1 rigor.

## Inputs (from Step 1)
- `metadata_rgb_multi4_with_split.csv`
- `split_manifest.json`
- `leakage_report.txt`

## Advanced scope
1. Skill classification with Transformer
2. Optional multi-task learning: skill (primary) + action (auxiliary)
3. Multi-seed mean/std reporting
4. Per-action skill metrics on test set
5. Gradient-based interpretation (saliency + integrated gradients)
6. Optional LOPO evaluation


## Professor Feedback Coverage

- **Class imbalance**: class-weighted loss + balanced sampler.
- **Stroke confound**: multi-task option (skill primary, action auxiliary) and per-action skill report.
- **Interpretation reliability**: gradient-based saliency and integrated gradients (not relying on attention as direct importance).
- **Reliability**: multi-seed summary and optional LOPO loop.


In [ ]:

# =============================

# 挂载 Google Drive（如果还没挂载，先运行这个）
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip -q install pandas numpy scikit-learn torch torchvision torchaudio

import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score, classification_report

print('Environment ready.')


Environment ready.


In [ ]:
# =============================
# Config
# =============================
SEED = 42

# If True, run multiple seeds and summarize mean/std
RUN_MULTI_SEED = True
SEEDS = [13, 42, 77]

# If True, run ablation without vel/acc features
RUN_ABLATION = True

# If True, run optional LOPO on best config (slow)
RUN_LOPO = True
LOPO_EPOCHS = 15

# Training params
EPOCHS = 30
BATCH_SIZE = 32
LR = 1e-3
AUX_LOSS_WEIGHT = 0.3  # weight for action auxiliary loss in multi-task

# Data paths
STEP1_DIR = Path('/content/drive/MyDrive/step1_outputs')
STEP1_META = STEP1_DIR / 'metadata_rgb_multi4_with_split.csv'
STEP1_MANIFEST = STEP1_DIR / 'split_manifest.json'
KPT_ROOT = Path('/content/drive/Shareddrives/ML2_Final/tennis_data')
OUT_DIR = Path('/content/step2_advanced_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Feature options
USE_VEL_ACC = True
USE_CONF = True

print('STEP1_META exists:', STEP1_META.exists())
print('KPT_ROOT exists:', KPT_ROOT.exists())
print('OUT_DIR:', OUT_DIR)


STEP1_META exists: True
KPT_ROOT exists: True
OUT_DIR: /content/step2_advanced_outputs


In [ ]:
# =============================
# Load Step1 metadata and verify split logic
# =============================
if not STEP1_META.exists():
    raise FileNotFoundError(f'Missing Step1 metadata: {STEP1_META}')
meta = pd.read_csv(STEP1_META)
required = {'video_name','subject_id','skill_binary','action_folder','split'}
missing = required - set(meta.columns)
if missing:
    raise ValueError(f'Step1 metadata missing columns: {missing}')
# Ensure canonical_action exists
if 'canonical_action' not in meta.columns:
    meta['canonical_action'] = meta['action_folder']
# Build keypoint paths
def to_npy_path(row):
    stem = row['video_name'].replace('.avi', '')
    kpt_folder = f"{row['action_folder']}_keypoints"  # ← 改这里
    return str((KPT_ROOT / kpt_folder / f'{stem}.npy').resolve())
meta['keypoint_npy'] = meta.apply(to_npy_path, axis=1)
exists_mask = meta['keypoint_npy'].apply(lambda p: Path(p).exists())
missing_df = meta.loc[~exists_mask, ['video_name','action_folder','keypoint_npy']]
if len(missing_df):
    missing_df.to_csv(OUT_DIR / 'missing_keypoint_files.csv', index=False)
    print(f'[WARN] missing keypoint files: {len(missing_df)}')
meta = meta.loc[exists_mask].copy().reset_index(drop=True)
# Split leakage check from metadata itself
train_subjects = set(meta.loc[meta['split']=='train','subject_id'])
val_subjects = set(meta.loc[meta['split']=='val','subject_id'])
test_subjects = set(meta.loc[meta['split']=='test','subject_id'])
assert len(train_subjects & val_subjects) == 0
assert len(train_subjects & test_subjects) == 0
assert len(val_subjects & test_subjects) == 0
print('Rows used:', len(meta))
print('Subjects:', meta['subject_id'].nunique())
print('Split counts:', meta['split'].value_counts(), sep='\n')
print('Skill counts:', meta['skill_binary'].value_counts(), sep='\n')
print('Action counts:', meta['action_folder'].value_counts(), sep='\n')

Rows used: 660
Subjects: 55
Split counts:
split
train    420
test     132
val      108
Name: count, dtype: int64
Skill counts:
skill_binary
0    372
1    288
Name: count, dtype: int64
Action counts:
action_folder
backhand         165
forehand_flat    165
kick_service     165
smash            165
Name: count, dtype: int64


In [ ]:
# =============================
# Feature extraction
# =============================

def extract_features(arr, use_vel_acc=True, use_conf=True, use_normalization=True):
    coords = arr[..., :2].astype(np.float32)
    conf = arr[..., 2].astype(np.float32)

    # body-centered normalization (can be disabled for ablation)
    if use_normalization:
        lsh = coords[:,5,:]
        rsh = coords[:,6,:]
        lhip = coords[:,11,:]
        rhip = coords[:,12,:]
        center = (lsh + rsh + lhip + rhip) / 4.0
        scale = np.linalg.norm(lsh - rhip, axis=1, keepdims=True) + 1e-6
        norm = (coords - center[:,None,:]) / scale[:,None,:]
    else:
        norm = coords  # raw pixel coords, no normalization

    parts = [norm.reshape(norm.shape[0], -1)]

    if use_vel_acc:
        vel = np.diff(norm, axis=0)
        vel = np.concatenate([vel, vel[-1:]], axis=0) if len(vel) else np.zeros_like(norm)
        acc = np.diff(vel, axis=0)
        acc = np.concatenate([acc, acc[-1:]], axis=0) if len(acc) else np.zeros_like(norm)
        parts.append(vel.reshape(vel.shape[0], -1))
        parts.append(acc.reshape(acc.shape[0], -1))

    if use_conf:
        parts.append(conf)

    feat = np.concatenate(parts, axis=1).astype(np.float32)
    return feat


def pad_or_truncate(seq, max_len):
    t, f = seq.shape
    if t >= max_len:
        return seq[:max_len]
    pad = np.zeros((max_len - t, f), dtype=np.float32)
    return np.concatenate([seq, pad], axis=0)


def build_arrays(meta_df, use_vel_acc=True, use_conf=True, use_normalization=True):
    seqs = [extract_features(np.load(p), use_vel_acc=use_vel_acc, use_conf=use_conf, use_normalization=use_normalization) for p in meta_df['keypoint_npy']]
    lengths = np.array([s.shape[0] for s in seqs])
    max_len = max(40, int(np.percentile(lengths, 95)))

    X = np.stack([pad_or_truncate(s, max_len) for s in seqs])

    action_to_id = {a:i for i,a in enumerate(sorted(meta_df['action_folder'].unique()))}
    y_action = meta_df['action_folder'].map(action_to_id).values.astype(int)
    y_skill = meta_df['skill_binary'].values.astype(int)

    idx_train = np.where(meta_df['split'].values=='train')[0]
    idx_val = np.where(meta_df['split'].values=='val')[0]
    idx_test = np.where(meta_df['split'].values=='test')[0]

    return X, y_skill, y_action, idx_train, idx_val, idx_test, action_to_id, max_len

In [ ]:
# =============================
# Transformer model + training utilities
# =============================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def safe_auc(y_true, prob, multiclass=False):
    if len(np.unique(y_true)) < 2:
        return float('nan')
    if multiclass:
        return float(roc_auc_score(y_true, prob, multi_class='ovr'))
    return float(roc_auc_score(y_true, prob))


def eval_metrics(y_true, y_pred, y_prob=None, multiclass=False):
    out = {
        'macro_f1': float(f1_score(y_true, y_pred, average='macro')),
        'balanced_acc': float(balanced_accuracy_score(y_true, y_pred)),
        'roc_auc': float('nan')
    }
    if y_prob is not None:
        out['roc_auc'] = safe_auc(y_true, y_prob, multiclass=multiclass)
    return out


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class TransformerSkillMultiTask(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=4, num_layers=2, dim_ff=256, dropout=0.2, num_actions=4):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos = PositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_ff, dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)

        self.skill_head = nn.Linear(d_model, 2)
        self.action_head = nn.Linear(d_model, num_actions)

    def forward(self, x):
        z = self.input_proj(x)
        z = self.pos(z)
        z = self.encoder(z)
        z = self.norm(z)
        pooled = z.mean(dim=1)
        skill_logits = self.skill_head(pooled)
        action_logits = self.action_head(pooled)
        return skill_logits, action_logits


def train_transformer(
    X, y_skill, y_action, idx_train, idx_val, idx_test,
    seed=42, epochs=30, lr=1e-3, batch_size=16,
    multitask=True, aux_weight=0.3
):
    set_seed(seed)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    Xtr = torch.tensor(X[idx_train], dtype=torch.float32)
    Xvl = torch.tensor(X[idx_val], dtype=torch.float32)
    Xte = torch.tensor(X[idx_test], dtype=torch.float32)

    ytr_skill = torch.tensor(y_skill[idx_train], dtype=torch.long)
    yvl_skill = torch.tensor(y_skill[idx_val], dtype=torch.long)
    yte_skill = torch.tensor(y_skill[idx_test], dtype=torch.long)

    ytr_action = torch.tensor(y_action[idx_train], dtype=torch.long)
    yvl_action = torch.tensor(y_action[idx_val], dtype=torch.long)
    yte_action = torch.tensor(y_action[idx_test], dtype=torch.long)

    # class weights for skill imbalance
    skill_counts = np.bincount(y_skill[idx_train], minlength=2).astype(np.float32)
    skill_w = skill_counts.sum() / (2.0 * np.maximum(skill_counts, 1.0))
    skill_weights = torch.tensor(skill_w, dtype=torch.float32).to(device)

    # balanced sampler for skill
    sample_w = np.array([1.0/skill_counts[c] for c in y_skill[idx_train]], dtype=np.float32)
    sampler = WeightedRandomSampler(weights=torch.tensor(sample_w), num_samples=len(sample_w), replacement=True)

    ds = TensorDataset(Xtr, ytr_skill, ytr_action)
    loader = DataLoader(ds, batch_size=batch_size, sampler=sampler)

    model = TransformerSkillMultiTask(input_dim=X.shape[-1], num_actions=len(np.unique(y_action))).to(device)

    crit_skill = nn.CrossEntropyLoss(weight=skill_weights)
    crit_action = nn.CrossEntropyLoss()
    opt = optim.Adam(model.parameters(), lr=lr)

    def infer(Xt, y_skill_t, y_action_t):
        model.eval()
        with torch.no_grad():
            skill_logits, action_logits = model(Xt.to(device))
            prob_skill = torch.softmax(skill_logits, dim=1)[:,1].cpu().numpy()
            pred_skill = skill_logits.argmax(dim=1).cpu().numpy()
            y_skill_np = y_skill_t.cpu().numpy()

            prob_action = torch.softmax(action_logits, dim=1).cpu().numpy()
            pred_action = action_logits.argmax(dim=1).cpu().numpy()
            y_action_np = y_action_t.cpu().numpy()

        m_skill = eval_metrics(y_skill_np, pred_skill, prob_skill, multiclass=False)
        m_action = eval_metrics(y_action_np, pred_action, prob_action, multiclass=True)
        return pred_skill, prob_skill, m_skill, pred_action, prob_action, m_action

    best_state = None
    best_val_f1 = -1.0

    for ep in range(1, epochs+1):
        model.train()
        for xb, yb_skill, yb_action in loader:
            xb = xb.to(device)
            yb_skill = yb_skill.to(device)
            yb_action = yb_action.to(device)

            opt.zero_grad()
            skill_logits, action_logits = model(xb)
            loss_skill = crit_skill(skill_logits, yb_skill)

            if multitask:
                loss_action = crit_action(action_logits, yb_action)
                loss = loss_skill + aux_weight * loss_action
            else:
                loss = loss_skill

            loss.backward()
            opt.step()

        _, _, val_skill_m, _, _, _ = infer(Xvl, yvl_skill, yvl_action)
        if val_skill_m['macro_f1'] > best_val_f1:
            best_val_f1 = val_skill_m['macro_f1']
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    out = {}
    for split, Xt, ys, ya in [('train',Xtr,ytr_skill,ytr_action),('val',Xvl,yvl_skill,yvl_action),('test',Xte,yte_skill,yte_action)]:
        ps, probs, ms, pa, proba, ma = infer(Xt, ys, ya)
        out[split] = {
            'pred_skill': ps,
            'prob_skill': probs,
            'metrics_skill': ms,
            'pred_action': pa,
            'prob_action': proba,
            'metrics_action': ma,
            'y_skill': ys.cpu().numpy(),
            'y_action': ya.cpu().numpy(),
        }

    return model, out


In [ ]:
# =============================
# Run advanced experiments (single-seed + multi-seed summary)
# =============================
from tqdm.notebook import tqdm

X, y_skill, y_action, idx_train, idx_val, idx_test, action_to_id, max_len = build_arrays(meta, use_vel_acc=USE_VEL_ACC, use_conf=USE_CONF)

experiments = [
    {'exp_name':'transformer_skill_only', 'multitask':False, 'use_vel_acc':True, 'use_conf':True, 'use_normalization':True},
    {'exp_name':'transformer_multitask', 'multitask':True, 'use_vel_acc':True, 'use_conf':True, 'use_normalization':True},
]
if RUN_ABLATION:
    experiments.append({'exp_name':'transformer_multitask_no_velacc', 'multitask':True, 'use_vel_acc':False, 'use_conf':True, 'use_normalization':True})
    experiments.append({'exp_name':'transformer_multitask_no_norm', 'multitask':True, 'use_vel_acc':True, 'use_conf':True, 'use_normalization':False})  # ← 新增 normalization ablation

rows = []
pred_store = {}
seed_list = SEEDS if RUN_MULTI_SEED else [SEED]

for exp in tqdm(experiments, desc='Experiments'):
    Xexp, ys, ya, it, iv, ite, amap, mlen = build_arrays(meta, use_vel_acc=exp['use_vel_acc'], use_conf=exp['use_conf'], use_normalization=exp.get('use_normalization', True))

    for sd in tqdm(seed_list, desc=f"Seeds [{exp['exp_name']}]", leave=False):
        model, out = train_transformer(
            Xexp, ys, ya, it, iv, ite,
            seed=sd, epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE,
            multitask=exp['multitask'], aux_weight=AUX_LOSS_WEIGHT
        )

        for split in ['train','val','test']:
            ms = out[split]['metrics_skill']
            ma = out[split]['metrics_action']
            rows.append({'exp_name':exp['exp_name'],'seed':sd,'split':split,'task':'skill', **ms})
            rows.append({'exp_name':exp['exp_name'],'seed':sd,'split':split,'task':'action', **ma})

        key = f"{exp['exp_name']}__seed{sd}"
        pred_store[key] = out['test']

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(OUT_DIR / 'metrics_transformer_all_runs.csv', index=False)

summary = metrics_df.groupby(['exp_name','task','split'])[['macro_f1','balanced_acc','roc_auc']].agg(['mean','std']).reset_index()
summary.to_csv(OUT_DIR / 'metrics_transformer_summary_mean_std.csv', index=False)

print('Saved:', OUT_DIR / 'metrics_transformer_all_runs.csv')
print('Saved:', OUT_DIR / 'metrics_transformer_summary_mean_std.csv')
summary.head(12)

Experiments:   0%|          | 0/4 [00:00<?, ?it/s]

Seeds [transformer_skill_only]:   0%|          | 0/3 [00:00<?, ?it/s]

Seeds [transformer_multitask]:   0%|          | 0/3 [00:00<?, ?it/s]

Seeds [transformer_multitask_no_velacc]:   0%|          | 0/3 [00:00<?, ?it/s]

Seeds [transformer_multitask_no_norm]:   0%|          | 0/3 [00:00<?, ?it/s]

Saved: /content/step2_advanced_outputs/metrics_transformer_all_runs.csv
Saved: /content/step2_advanced_outputs/metrics_transformer_summary_mean_std.csv


exp_name    task  split  macro_f1            \
                                                      mean       std   
0           transformer_multitask  action   test  0.601646  0.047408   
1           transformer_multitask  action  train  0.727533  0.019821   
2           transformer_multitask  action    val  0.753720  0.040920   
3           transformer_multitask   skill   test  0.824922  0.042194   
4           transformer_multitask   skill  train  0.869400  0.060873   
5           transformer_multitask   skill    val  0.902558  0.014669   
6   transformer_multitask_no_norm  action   test  0.498451  0.054495   
7   transformer_multitask_no_norm  action  train  0.594126  0.103794   
8   transformer_multitask_no_norm  action    val  0.541983  0.115307   
9   transformer_multitask_no_norm   skill   test  0.764247  0.085231   
10  transformer_multitask_no_norm   skill  train  0.757308  0.077019   
11  transformer_multitask_no_norm   skill    val  0.727614  0.060525   

   balanced_acc             roc_auc            
           mean       std      mean       std  
0      0.611111  0.041724  0.874809  0.013505  
1      0.734127  0.019245  0.934041  0.013004  
2      0.762346  0.037421  0.948255  0.004812  
3      0.833333  0.036031  0.918519  0.018296  
4      0.881250  0.051637  0.970378  0.010887  
5      0.900694  0.015356  0.955324  0.027887  
6      0.507576  0.054630  0.790889  0.054635  
7      0.608730  0.103492  0.860806  0.060382  
8      0.561728  0.117972  0.831885  0.067502  
9      0.768056  0.087147  0.826466  0.123473  
10     0.760880  0.075998  0.841296  0.098371  
11     0.729167  0.063053  0.818171  0.072537

In [ ]:
# =============================
# Per-action skill metrics from best advanced run (test)
# =============================

# choose best run by val skill macro_f1
val_skill = metrics_df[(metrics_df['split']=='val') & (metrics_df['task']=='skill')]
best_row = val_skill.sort_values('macro_f1', ascending=False).iloc[0]
best_exp = best_row['exp_name']
best_seed = int(best_row['seed'])
key = f"{best_exp}__seed{best_seed}"

best_test = pred_store[key]
test_meta = meta.iloc[idx_test].reset_index(drop=True)

pred_df = pd.DataFrame({
    'video_name': test_meta['video_name'].values,
    'subject_id': test_meta['subject_id'].values,
    'action_folder': test_meta['action_folder'].values,
    'y_true_skill': best_test['y_skill'],
    'y_pred_skill': best_test['pred_skill'],
    'prob_expert': best_test['prob_skill'],
    'best_exp': best_exp,
    'best_seed': best_seed,
})
pred_df.to_csv(OUT_DIR / 'predictions_test_best_transformer_skill.csv', index=False)

rows = []
for act in sorted(pred_df['action_folder'].unique()):
    sub = pred_df[pred_df['action_folder']==act]
    m = eval_metrics(sub['y_true_skill'].values, sub['y_pred_skill'].values, sub['prob_expert'].values, multiclass=False)
    rows.append({'action_folder':act, 'n_test':len(sub), **m, 'best_exp':best_exp, 'best_seed':best_seed})

per_action_df = pd.DataFrame(rows)
per_action_df.to_csv(OUT_DIR / 'per_action_skill_metrics_best_transformer.csv', index=False)

print('Best run:', best_exp, 'seed', best_seed)
print(per_action_df)


Best run: transformer_skill_only seed 77
   action_folder  n_test  macro_f1  balanced_acc   roc_auc  \
0       backhand      33  0.848485      0.855556  0.981481   
1  forehand_flat      33  0.696691      0.700000  0.825926   
2   kick_service      33  0.818015      0.822222  0.885185   
3          smash      33  0.787879      0.794444  0.825926   

                 best_exp  best_seed  
0  transformer_skill_only         77  
1  transformer_skill_only         77  
2  transformer_skill_only         77  
3  transformer_skill_only         77  


In [ ]:
# =============================
# Gradient-based interpretation — 全量test集聚合 + 关节分组
# =============================

# Joint group mapping (COCO 17-keypoint order)
JOINT_NAMES = [
    'nose','left_eye','right_eye','left_ear','right_ear',
    'left_shoulder','right_shoulder','left_elbow','right_elbow',
    'left_wrist','right_wrist','left_hip','right_hip',
    'left_knee','right_knee','left_ankle','right_ankle'
]
JOINT_GROUPS = {
    'upper_limb':   [5,6,7,8,9,10],     # shoulders, elbows, wrists
    'lower_limb':   [11,12,13,14,15,16], # hips, knees, ankles
    'trunk_head':   [0,1,2,3,4],         # nose, eyes, ears
}

def joint_group_attribution(ig_or_sal, n_joints=17):
    result = {}
    for grp, jidxs in JOINT_GROUPS.items():
        cols = []
        for jj in jidxs:
            cols += [jj*2, jj*2+1]
            cols += [34+jj*2, 34+jj*2+1]
            cols += [68+jj*2, 68+jj*2+1]
        cols = [c for c in cols if c < ig_or_sal.shape[1]]
        result[grp] = float(np.abs(ig_or_sal[:, cols]).mean())
    return result

def integrated_gradients(model, x, baseline=None, steps=24):
    if baseline is None:
        baseline = torch.zeros_like(x)
    alphas = torch.linspace(0, 1, steps).to(x.device)
    grads = []
    for a in alphas:
        xi = (baseline + a * (x - baseline)).detach().requires_grad_(True)
        logits, _ = model(xi)
        s = logits[0, 1]
        g = torch.autograd.grad(s, xi, retain_graph=False)[0]
        grads.append(g.detach())
    avg_grad = torch.mean(torch.stack(grads), dim=0)
    ig = (x - baseline) * avg_grad
    return ig.detach().cpu().numpy()[0]

# Re-train best model
best_cfg = [e for e in experiments if e['exp_name']==best_exp][0]
Xb, ysb, yab, it, iv, ite, amap, mlen = build_arrays(
    meta,
    use_vel_acc=best_cfg['use_vel_acc'],
    use_conf=best_cfg['use_conf'],
    use_normalization=best_cfg.get('use_normalization', True)
)
model_best, out_best = train_transformer(
    Xb, ysb, yab, it, iv, ite,
    seed=best_seed, epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE,
    multitask=best_cfg['multitask'], aux_weight=AUX_LOSS_WEIGHT
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_best.to(device)
model_best.eval()

# 按skill label分组test indices
test_y_skill = ysb[ite]
test_expert_idx   = np.where(test_y_skill == 1)[0]
test_beginner_idx = np.where(test_y_skill == 0)[0]
print(f'Test samples: {len(ite)} total | expert={len(test_expert_idx)} | beginner={len(test_beginner_idx)}')

# ── 全量 Saliency 聚合 ──────────────────────────────────────────────
def compute_saliency_batch(model, X_arr, local_indices, device):
    sals = []
    for li in local_indices:
        x_s = torch.tensor(X_arr[ite][li:li+1], dtype=torch.float32).to(device)
        x_s.requires_grad_(True)
        logits, _ = model(x_s)
        logits[0, 1].backward()
        sal = x_s.grad.detach().abs().cpu().numpy()[0]
        sals.append(sal)
        model.zero_grad()
    return np.array(sals)

print('Computing saliency for all test samples...')
sal_expert   = compute_saliency_batch(model_best, Xb, test_expert_idx, device)
sal_beginner = compute_saliency_batch(model_best, Xb, test_beginner_idx, device)

mean_sal_expert   = sal_expert.mean(axis=0)
mean_sal_beginner = sal_beginner.mean(axis=0)

sal_time_expert   = mean_sal_expert.mean(axis=1)
sal_time_beginner = mean_sal_beginner.mean(axis=1)

sal_time_df = pd.DataFrame({
    'time_idx': np.arange(len(sal_time_expert)),
    'saliency_expert_mean':   sal_time_expert,
    'saliency_beginner_mean': sal_time_beginner,
})
sal_time_df.to_csv(OUT_DIR / 'saliency_time_profile_aggregated.csv', index=False)

# joint-group
sal_grp_rows = []
for grp in JOINT_GROUPS:
    sal_grp_rows.append({
        'group': grp,
        'saliency_expert':   joint_group_attribution(mean_sal_expert)[grp],
        'saliency_beginner': joint_group_attribution(mean_sal_beginner)[grp],
    })
sal_group_df = pd.DataFrame(sal_grp_rows)
sal_group_df.to_csv(OUT_DIR / 'saliency_joint_group_aggregated.csv', index=False)
print('Saliency joint groups:\n', sal_group_df)

# phase-level
T = len(sal_time_expert)
phase_bounds = {
    'preparation':    (0, T//3),
    'contact':        (T//3, 2*T//3),
    'follow_through': (2*T//3, T)
}
phase_rows = []
for phase, (s, e) in phase_bounds.items():
    phase_rows.append({
        'phase': phase,
        'saliency_expert':   float(sal_time_expert[s:e].mean()),
        'saliency_beginner': float(sal_time_beginner[s:e].mean()),
    })
sal_phase_df = pd.DataFrame(phase_rows)
sal_phase_df.to_csv(OUT_DIR / 'saliency_phase_level.csv', index=False)
print('Saliency phases:\n', sal_phase_df)

# ── 全量 Integrated Gradients 聚合 ──────────────────────────────────
print('Computing IG for all test samples...')
ig_expert_list   = []
ig_beginner_list = []

for li in test_expert_idx:
    x_in = torch.tensor(Xb[ite][li:li+1], dtype=torch.float32).to(device)
    ig = integrated_gradients(model_best, x_in, baseline=torch.zeros_like(x_in), steps=24)
    ig_expert_list.append(ig)

for li in test_beginner_idx:
    x_in = torch.tensor(Xb[ite][li:li+1], dtype=torch.float32).to(device)
    ig = integrated_gradients(model_best, x_in, baseline=torch.zeros_like(x_in), steps=24)
    ig_beginner_list.append(ig)

mean_ig_expert   = np.mean(ig_expert_list, axis=0)
mean_ig_beginner = np.mean(ig_beginner_list, axis=0)

ig_time_df = pd.DataFrame({
    'time_idx': np.arange(mean_ig_expert.shape[0]),
    'ig_expert_mean':   np.abs(mean_ig_expert).mean(axis=1),
    'ig_beginner_mean': np.abs(mean_ig_beginner).mean(axis=1),
})
ig_time_df.to_csv(OUT_DIR / 'integrated_gradients_time_profile_aggregated.csv', index=False)

# joint-group
ig_grp_rows = []
for grp in JOINT_GROUPS:
    ig_grp_rows.append({
        'group': grp,
        'ig_expert':   joint_group_attribution(np.abs(mean_ig_expert))[grp],
        'ig_beginner': joint_group_attribution(np.abs(mean_ig_beginner))[grp],
    })
ig_group_df = pd.DataFrame(ig_grp_rows)
ig_group_df.to_csv(OUT_DIR / 'integrated_gradients_joint_group_aggregated.csv', index=False)
print('IG joint groups:\n', ig_group_df)

# phase-level
phase_ig_rows = []
for phase, (s, e) in phase_bounds.items():
    phase_ig_rows.append({
        'phase': phase,
        'ig_expert':   float(np.abs(mean_ig_expert[s:e]).mean()),
        'ig_beginner': float(np.abs(mean_ig_beginner[s:e]).mean()),
    })
ig_phase_df = pd.DataFrame(phase_ig_rows)
ig_phase_df.to_csv(OUT_DIR / 'integrated_gradients_phase_level.csv', index=False)
print('IG phases:\n', ig_phase_df)

print('\nAll interpretation outputs saved.')

Test samples: 132 total | expert=60 | beginner=72
Computing saliency for all test samples...
Saliency joint groups:
         group  saliency_expert  saliency_beginner
0  upper_limb         0.009582           0.009196
1  lower_limb         0.011045           0.010818
2  trunk_head         0.010384           0.010008
Saliency phases:
             phase  saliency_expert  saliency_beginner
0     preparation         0.011333           0.010366
1         contact         0.014071           0.014303
2  follow_through         0.002913           0.002711
Computing IG for all test samples...
IG joint groups:
         group  ig_expert  ig_beginner
0  upper_limb   0.000514     0.000567
1  lower_limb   0.000454     0.000869
2  trunk_head   0.000334     0.000435
IG phases:
             phase  ig_expert  ig_beginner
0     preparation   0.000781     0.000997
1         contact   0.000905     0.001388
2  follow_through   0.000076     0.000134

All interpretation outputs saved.


In [ ]:
# =============================
# LOPO — Leave-One-Player-Out (required by protocol)
# Improved: val split uses random sample instead of fixed first 1/6
# =============================
if RUN_LOPO:
    best_cfg_lopo = [e for e in experiments if e['exp_name'] == best_exp][0]
    X_lopo, y_skill_lopo, y_action_lopo, _, _, _, _, _ = build_arrays(
        meta,
        use_vel_acc=best_cfg_lopo['use_vel_acc'],
        use_conf=best_cfg_lopo['use_conf'],
        use_normalization=best_cfg_lopo.get('use_normalization', True)
    )

    subj_list = sorted(meta['subject_id'].unique())
    lopo_rows = []
    rng = np.random.default_rng(SEED)

    print(f'Running LOPO over {len(subj_list)} subjects with {LOPO_EPOCHS} epochs each...')

    for sid in subj_list:
        test_mask  = (meta['subject_id'] == sid).values
        train_mask = ~test_mask

        idx_te     = np.where(test_mask)[0]
        idx_tr_all = np.where(train_mask)[0]

        if len(idx_te) == 0:
            continue
        if len(np.unique(y_skill_lopo[idx_te])) < 1:
            continue

        # random 15% of remaining subjects as val set
        train_subj = np.array(sorted(set(meta.iloc[idx_tr_all]['subject_id'].unique())))
        n_val_subj = max(1, int(len(train_subj) * 0.15))
        val_subj   = set(rng.choice(train_subj, size=n_val_subj, replace=False).tolist())

        idx_val_l = np.array([i for i in idx_tr_all if meta.iloc[i]['subject_id'] in val_subj])
        idx_tr_l  = np.array([i for i in idx_tr_all if meta.iloc[i]['subject_id'] not in val_subj])

        if len(idx_tr_l) == 0 or len(idx_val_l) == 0:
            continue

        model_l, out_l = train_transformer(
            X_lopo, y_skill_lopo, y_action_lopo,
            idx_tr_l, idx_val_l, idx_te,
            seed=SEED, epochs=LOPO_EPOCHS, lr=LR, batch_size=BATCH_SIZE,
            multitask=True, aux_weight=AUX_LOSS_WEIGHT
        )

        m = out_l['test']['metrics_skill']
        lopo_rows.append({
            'heldout_subject': sid,
            'skill_label': int(meta.loc[meta['subject_id']==sid, 'skill_binary'].iloc[0]),
            'n_test_seq': len(idx_te),
            **m
        })
        print(f'  {sid}: macro_f1={m["macro_f1"]:.3f}  bal_acc={m["balanced_acc"]:.3f}')

    lopo_df = pd.DataFrame(lopo_rows)
    lopo_df.to_csv(OUT_DIR / 'lopo_skill_metrics_transformer.csv', index=False)

    lopo_summary = lopo_df[['macro_f1','balanced_acc','roc_auc']].agg(['mean','std'])
    lopo_summary.to_csv(OUT_DIR / 'lopo_skill_summary.csv')

    print('\nLOPO Summary:')
    print(lopo_summary)
    print('Saved LOPO metrics:', OUT_DIR / 'lopo_skill_metrics_transformer.csv')

else:
    print('RUN_LOPO=False -> skipped LOPO block')

Running LOPO over 55 subjects with 15 epochs each...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  p1: macro_f1=0.478  bal_acc=0.917


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


  p10: macro_f1=1.000  bal_acc=1.000


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


  p7: macro_f1=1.000  bal_acc=1.000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  p8: macro_f1=0.368  bal_acc=0.583
  p9: macro_f1=0.333  bal_acc=0.500

LOPO Summary:
      macro_f1  balanced_acc  roc_auc
mean  0.577798      0.789394      NaN
std   0.280377      0.209014      NaN
Saved LOPO metrics: /content/step2_advanced_outputs/lopo_skill_metrics_transformer.csv


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [ ]:
# =============================
# Final packaging
# =============================
handoff_note = """# Step 2 Advanced Handoff Note

## Included
- Transformer skill-only and multi-task variants
- Multi-seed mean/std summary
- Per-action skill breakdown on test
- Gradient saliency + integrated gradients sample outputs
- Optional LOPO support

## Professor feedback mapping
1. Class imbalance handled with weighted loss + sampler.
2. Stroke confound addressed via auxiliary action head and per-action skill metrics.
3. Interpretation uses gradient-based methods (not attention-only claims).
4. Reliability strengthened via multi-seed summary and optional LOPO.
"""
(OUT_DIR / 'step2_advanced_handoff_note.md').write_text(handoff_note)

# merge quick report table from summary
if (OUT_DIR / 'metrics_transformer_summary_mean_std.csv').exists():
    quick = pd.read_csv(OUT_DIR / 'metrics_transformer_summary_mean_std.csv')
    quick.to_csv(OUT_DIR / 'final_table_transformer_mean_std.csv', index=False)

print('Result files:')
for p in sorted(OUT_DIR.glob('*')):
    print('-', p.name)

import shutil
from google.colab import files
zip_path = '/content/step2_advanced_outputs.zip'
shutil.make_archive('/content/step2_advanced_outputs', 'zip', OUT_DIR)
print('Created:', zip_path)
files.download(zip_path)


Result files:
- final_table_transformer_mean_std.csv
- integrated_gradients_joint_group_aggregated.csv
- integrated_gradients_phase_level.csv
- integrated_gradients_time_profile_aggregated.csv
- lopo_skill_metrics_transformer.csv
- lopo_skill_summary.csv
- metrics_transformer_all_runs.csv
- metrics_transformer_summary_mean_std.csv
- per_action_skill_metrics_best_transformer.csv
- predictions_test_best_transformer_skill.csv
- saliency_joint_group_aggregated.csv
- saliency_phase_level.csv
- saliency_time_profile_aggregated.csv
- step2_advanced_handoff_note.md
Created: /content/step2_advanced_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>